In [0]:
dbutils.widgets.removeAll()

In [0]:
# Databricks notebook source
# DBTITLE 1,Setup logging
import logging
from datetime import datetime
import os

# Create logs directory if it doesn't exist
log_dir = "/Workspace/Users/logi@openhealthagents.org/edi/logs"
dbutils.fs.mkdirs(f"file:{log_dir}")

# Create log filename with timestamp
log_filename = f"{log_dir}/member_person_bridge_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        # sends log to the permanent file
        logging.FileHandler(log_filename.replace('file:', '')),
        # sends log directly to the terminal
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("Member Person Bridge Processing Started")
logger.info(f"Log file: {log_filename}")

In [0]:
# Databricks notebook source
# DBTITLE 1,Configuration - Auto-detect source

# Source: Delta table in Volumes
sourcePath = "/Volumes/edi/bronze/consolidation/member/"
goldTable = "edi.gold.member_person_bridge"

logger.info("Configuration loaded:")
logger.info(f"  Source Path: {sourcePath}")
logger.info(f"  Gold Table: {goldTable}")

In [0]:
# Databricks notebook source
# DBTITLE 1,Install libraries
%pip install recordlinkage -q

In [0]:
# Databricks notebook source
# DBTITLE 1,Import libraries
from pyspark.sql.functions import date_format, monotonically_increasing_id, sha2, concat_ws, col, row_number, substring, lower, coalesce, upper, trim, regexp_replace
from pyspark.sql.window import Window
import pandas as pd
import recordlinkage
from pyspark.sql.types import StructType,StructField, StringType, IntegerType, LongType

In [0]:
# Databricks notebook source
# DBTITLE 1,Load source data
def LoadSource(sourcePath):
    logger.info(f"Starting LoadSource from: {sourcePath}")
    dfMBPGold = spark.read.format("delta").load(sourcePath)
    logger.info(f"Raw records loaded: {dfMBPGold.count()}")
    windowPartition = Window.partitionBy(col("FileId")).orderBy(col("RecordHash").desc())
    sparkMem_df = dfMBPGold.distinct() \
        .withColumn("RecordHash", sha2(concat_ws("||", *dfMBPGold.columns),256)) \
        .withColumn("RowNumber", row_number().over(windowPartition)) \
        .withColumn("UniqueRecord", concat_ws("-",col("FileID"),col("RowNumber"))) \
        .withColumn("BeneficiaryID", upper(trim(col("BeneficiaryID")))) \
        .withColumn("PlanMemberID", upper(trim(col("PlanMemberID")))) \
        .withColumn("UniquePersonKey", upper(trim(col("UniquePersonKey")))) \
        .withColumn("LastName", upper(trim(col("LastName")))) \
        .withColumn("FirstName", upper(trim(col("FirstName")))) \
        .withColumn("LastInitial", upper(substring(trim(col("LastName")),1,1))) \
        .withColumn("FirstInitial", upper(substring(trim(col("FirstName")),1,1))) \
        .withColumn("DateofBirthFormatted",date_format(trim(col("DateofBirth")),"yyyyMMdd")) \
        .withColumn("PhoneNumber", trim(col("PhoneNumber"))) \
        .withColumn("PhoneNumberFormatted", regexp_replace(trim(col("PhoneNumber")),"[^0-9]","")) \
        .withColumn("PermanentAddressLine1", upper(trim(col("PermanentAddressLine1")))) \
        .select("UniqueRecord","FileLayoutID","FileID","RowNumber","LastName","FirstName","Gender","PhoneNumber","PermanentAddressLine1","DateofBirthFormatted","PlanMemberID","BeneficiaryID","UniquePersonKey","LastInitial","FirstInitial","PhoneNumberFormatted")
    logger.info(f"Processed records after transformations: {sparkMem_df.count()}")
    logger.info("LoadSource completed successfully")
    return sparkMem_df

In [0]:
# Databricks notebook source
# DBTITLE 1,Define matching rules
RuleMBIColumns = ["BeneficiaryID","FirstInitial","LastInitial","DateofBirthFormatted","PhoneNumberFormatted","PermanentAddressLine1",11]
RulePMIDColumns = ["PlanMemberID","FirstInitial","LastInitial","DateofBirthFormatted","PhoneNumberFormatted","PermanentAddressLine1",11]
RuleUPKColumns = ["UniquePersonKey","FirstInitial","LastInitial","DateofBirthFormatted","PhoneNumberFormatted","PermanentAddressLine1",11]
RuleOtherColumns = ["DateofBirthFormatted","FirstName", "LastName","PhoneNumberFormatted","PermanentAddressLine1",14]
RulesAll = [RuleMBIColumns, RulePMIDColumns, RuleUPKColumns, RuleOtherColumns]
CompareColumns = ["LastInitial", "FirstInitial", "LastName", "FirstName", "BeneficiaryID", "PlanMemberID", "UniquePersonKey", "DateofBirthFormatted", "PhoneNumberFormatted", "PermanentAddressLine1"]

In [0]:
# Databricks notebook source
# DBTITLE 1,Record linkage - Compare function
def RulesToCompare(rules, pandasMem_df):
    logger.info("Starting RulesToCompare function")
    matchesAllRules_df = pd.DataFrame()
    for i, lst in enumerate(rules, 1):
        indexer = recordlinkage.Index()
        indexer.block(lst[0])
        candidatesBlock = indexer.index(pandasMem_df)
        logger.info(f"Rule {i} ({lst[0]}): Found {len(candidatesBlock)} candidate pairs")
        compareBlock = recordlinkage.Compare()
        threshold = lst[-1]
        for col in CompareColumns:
            compareBlock.exact(col, col, label=str(col))
        features = compareBlock.compute(candidatesBlock, pandasMem_df)
        features[lst[0]] = features[lst[0]].apply(lambda x: x*10)
        matchesRule = features[features[lst[:-1]].sum(axis=1) >= threshold]
        matchesRule_df = matchesRule.index.to_frame()
        logger.info(f"Rule {i}: {len(matchesRule)} matches found (threshold: {threshold})")
        logger.info(f"DEBUG Rule {i}: matchesRule_df shape: {matchesRule_df.shape}, columns: {list(matchesRule_df.columns)}")
        if len(matchesRule_df) > 0:
            logger.info(f"DEBUG Rule {i}: First row:\n{matchesRule_df.head(1)}")
        matchesAllRules_df = pd.concat([matchesAllRules_df, matchesRule_df])
    logger.info(f"RulesToCompare completed. Total matches across all rules: {len(matchesAllRules_df)}")
    return matchesAllRules_df

# COMMAND ----------
# DBTITLE 1,Record linkage - Main linking function
def RunLinking(sparkMem_df):
    logger.info("=" * 50)
    logger.info("STEP 1: Entering RunLinking function")
    logger.info("=" * 50)
    
    # Convert to Pandas
    pandasMem_df = sparkMem_df.toPandas()
    logger.info(f"Step 1a: Converted Spark DataFrame to Pandas DataFrame.")
    logger.info(f"         Initial Pandas Shape: {pandasMem_df.shape} (Rows, Columns)")
    
    # Set index with drop=False to avoid losing the UniqueRecord column
    pandasMem_df = pandasMem_df.set_index("UniqueRecord", drop=False)
    logger.info("Step 1b: Set 'UniqueRecord' as the index (with drop=False).")
    logger.info(f"         Columns currently available: {list(pandasMem_df.columns)}")
    
    # Run Record Linkage rules
    logger.info("STEP 2: Evaluating Record Linkage rules via RulesToCompare...")
    dfCombined = RulesToCompare(RulesAll, pandasMem_df)
    logger.info(f"Step 2a: RulesToCompare completed.")
    logger.info(f"         dfCombined structure -> Shape: {dfCombined.shape}, Empty: {dfCombined.empty}")
    
    # Handle the empty matches edge case safely
    if dfCombined.empty:
        logger.info("Step 2b: NO MATCHES FOUND across any rules. Falling back to self-assignment path.")
        finalPandas_df = pandasMem_df.copy()
        finalPandas_df['MatchID'] = finalPandas_df['UniqueRecord']
        logger.info(f"         Assigned MatchID = UniqueRecord. Final shape to Spark: {finalPandas_df.shape}")
        return spark.createDataFrame(finalPandas_df.astype(str))
    
    # Clean the MultiIndex columns from recordlinkage
    logger.info("STEP 3: Cleaning MultiIndex column structural headers")
    logger.info(f"         Original dfCombined columns before clean: {list(dfCombined.columns)}")
    dfCombined.columns = [0, 1]
    logger.info("         Successfully reset dfCombined columns to strict layout: [0, 1]")
    
    # Create bidirectional match sets
    logger.info("STEP 4: Reshaping pairs for directional alignment (A->B and B->A)")
    dfMatchedColumnA = dfCombined.rename(columns={0:"A", 1:"B"})
    dfMatchedColumnB = dfCombined.rename(columns={0:"B", 1:"A"})
    logger.info(f"         dfMatchedColumnA shape: {dfMatchedColumnA.shape} | dfMatchedColumnB shape: {dfMatchedColumnB.shape}")
    
    dfMatched = pd.concat([dfMatchedColumnA, dfMatchedColumnB], ignore_index=True)
    logger.info(f"         Combined shape before deduplication: {dfMatched.shape}")
    dfMatched.drop_duplicates(inplace=True)
    logger.info(f"         Combined shape after deduplication: {dfMatched.shape}")
    
    # Build a complete cross-reference set
    logger.info("STEP 5: Formatting cross-reference master table (Record <-> Match)")
    matchesAll_df = pd.concat([
        dfMatched[["A", "B"]].rename(columns={"A": "Record", "B": "Match"}),
        dfMatched[["B", "A"]].rename(columns={"B": "Record", "A": "Match"})
    ]).reset_index(drop=True)
    logger.info(f"         Master pairing shape (excluding identity pairs): {matchesAll_df.shape}")
    
    # Inject self-identity mappings (Record maps to itself)
    logger.info("Step 5b: Generating and appending identity pairs (Record maps to self)")
    matchesSame_df = matchesAll_df[["Record", "Record"]].copy()
    matchesSame_df.columns = ['Record', 'Match']
    
    matchesAll_df = pd.concat([matchesAll_df, matchesSame_df], ignore_index=True)
    matchesAll_df.drop_duplicates(inplace=True)
    logger.info(f"         Master pairing shape after appending identity links: {matchesAll_df.shape}")
    
    # Group by A to keep a representative linkage anchor row
    logger.info("STEP 6: Generating single anchor map grouping via Column 'A'")
    distinctRow = dfMatched.groupby('A').head(1).drop('B', axis=1)
    logger.info(f"         Anchor map frame shape: {distinctRow.shape}")
    
    matchesAll_df = pd.merge(matchesAll_df, distinctRow, how="left", left_on="Record", right_on="A").drop('A', axis=1)
    logger.info(f"         Master table shape after anchor reference integration: {matchesAll_df.shape}")
    
    # Merge back to parent DataFrame to extract safe Tracking IDs
    logger.info("STEP 7: Performing lookups against parent frame using Index references")
    matchesAll_df = pd.merge(matchesAll_df, pandasMem_df["UniqueRecord"], left_on="Record", right_index=True).rename(columns={"UniqueRecord":"RecordID"})
    matchesAll_df = pd.merge(matchesAll_df, pandasMem_df["UniqueRecord"], left_on="Match", right_index=True).rename(columns={"UniqueRecord":"MatchID"})
    logger.info(f"         Post-lookup tracking frame state -> Shape: {matchesAll_df.shape}")
    
    # Group individual matches into sorted arrays
    logger.info("STEP 8: Aggregating structural match arrays per RecordID (Groupby List consolidation)")
    matched_df = matchesAll_df.groupby("RecordID") \
            .agg({"MatchID": lambda x: list(pd.unique(x))}) \
            .reset_index()
    matched_df["MatchID"] = matched_df["MatchID"].apply(lambda x: sorted(x))
    logger.info(f"         Consolidated array dictionary shape: {matched_df.shape}")
    
    # Deduplicate rules tracking information
    logger.info("STEP 9: Stripping layout metadata metrics to create clean joining payload")
    matchesAllModified = matchesAll_df.groupby("RecordID").head(1)
    drop_cols = ['MatchID', 'Match', 'Record']
    existing_drops = [c for c in drop_cols if c in matchesAllModified.columns]
    matchesAllModified = matchesAllModified.drop(columns=existing_drops)
    if 'index' in matchesAllModified.columns:
        matchesAllModified = matchesAllModified.drop(columns=['index'])
    logger.info(f"         Payload frame shape ready for join: {matchesAllModified.shape}")
    
    newDFToMatch = pd.merge(matched_df, matchesAllModified, how="left", on="RecordID")
    logger.info(f"         Final cluster payload mapping shape: {newDFToMatch.shape}")
    
    # Assemble final master dataset
    logger.info("STEP 10: Merging linkage clusters back to master source pandas frame")
    # Create a clean copy of the columns without the index constraint to eliminate ambiguity
    finalPandas_df = pandasMem_df.reset_index(drop=True).merge(newDFToMatch, left_on="UniqueRecord", right_on="RecordID", how="left")
    logger.info(f"         Pre-fill master shape: {finalPandas_df.shape}")
    
    # Handle gaps for records with no matched cluster companions
    finalPandas_df['MatchID'] = finalPandas_df['MatchID'].fillna(finalPandas_df['UniqueRecord'])
    logger.info("         Filled missing match groups with self-referential UniqueRecord fallback values.")
    
    logger.info("=" * 50)
    logger.info(f"RunLinking COMPLETED SUCCESSFULLY. Final records to Spark: {len(finalPandas_df)}")
    logger.info("=" * 50)
    
    return spark.createDataFrame(finalPandas_df.astype(str))

In [0]:
# Databricks notebook source
# DBTITLE 1,Define SQL transformations
finalSQL = """
WITH BISPersonWithIdentifiers AS(
SELECT 
   UniqueRecord,FileLayoutID,FileId,RowNumber,LastName,FirstName,DateofBirthFormatted AS DateOfBirth,Gender,PermanentAddressLine1,PhoneNumber,PlanMemberID,BeneficiaryID,UniquePersonKey
  ,case when instr(MatchID,',')=0 then MatchID else substr(MatchID,2,instr(MatchID,',')) end as MatchID
  ,ROW_NUMBER() OVER(PARTITION BY (case when instr(MatchID,',')=0 then MatchID else substr(MatchID,2,instr(MatchID,',')) end) ORDER BY FileId ASC, RowNumber ASC) AS FirstPersonIdentifier
  ,ROW_NUMBER() OVER(PARTITION BY (case when instr(MatchID,',')=0 then MatchID else substr(MatchID,2,instr(MatchID,',')) end) ORDER BY FileId DESC, RowNumber DESC) AS CurrentPersonIdentifier
FROM BISCompletePersonTable
)
,MemberPersonBridge AS(
SELECT 
   fp.UniqueRecord AS BISInternalPersonID
  ,CASE WHEN cp.CurrentPersonIdentifier = 1 THEN 1 ELSE 0 END AS IsCurrent
  ,cp.UniqueRecord,cp.FileLayoutID,cp.FileId,cp.RowNumber,cp.LastName,cp.FirstName,cp.DateOfBirth,cp.Gender,cp.PermanentAddressLine1,cp.PhoneNumber,cp.PlanMemberID,cp.BeneficiaryID,cp.UniquePersonKey,cp.MatchID
FROM BISPersonWithIdentifiers cp
  LEFT JOIN BISPersonWithIdentifiers fp ON cp.MatchId = fp.MatchId AND fp.FirstPersonIdentifier = 1
)
,MemberPersonBridge_CurrPlanMbr AS(
SELECT 
   BISInternalPersonID,IsCurrent,UniqueRecord,FileLayoutID,FileId,RowNumber,LastName,FirstName,DateOfBirth,Gender,PermanentAddressLine1,PhoneNumber,PlanMemberID,BeneficiaryID
  ,ifnull(nullif(PlanMemberID,'None'),'') AS PlanMemberIdModified
  ,ifnull(nullif(UniquePersonKey,'None'),'') AS UniquePersonKeyModified
  ,UniquePersonKey,MatchID
  ,case when ifnull(PlanMemberID,'None')='None' then null when row_number() over(partition by PlanMemberID order by COALESCE(FileId,0) desc, COALESCE(RowNumber,0) desc) = 1 then 1 else 0 end as IsCurrentPlanMemberID
  ,case when ifnull(UniquePersonKey,'None')='None' then null when row_number() over(partition by UniquePersonKey order by COALESCE(FileId,0) desc, COALESCE(RowNumber,0) desc) = 1 then 1 else 0 end as IsCurrentUniquePersonKey
  ,case when BISInternalPersonID = UniqueRecord then 1 else 0 end AS IsOriginalMemberID 
FROM MemberPersonBridge
)
,PUModPop AS(
SELECT *
    ,CASE WHEN PlanMemberIdModified <> '' THEN 1 ELSE 0 END AS IsPlanMemberIdPopulated
    ,CASE WHEN UniquePersonKeyModified <> '' THEN 1 ELSE 0 END AS IsUniquePersonKeyModifiedPopulated
    ,concat(PlanMemberIdModified,'-',UniquePersonKeyModified) AS PMUP
FROM MemberPersonBridge_CurrPlanMbr
)
,Final AS (
SELECT *
    ,CAST(COALESCE(IsCurrentPlanMemberID,IsCurrentUniquePersonKey) AS STRING) AS IsCurrentPMUP
FROM PUModPop
)
SELECT 
   BISInternalPersonID
  ,IsCurrent
  ,UniqueRecord
  ,CAST(FileLayoutID AS INT) AS FileLayoutID
  ,FileId
  ,LastName
  ,FirstName
  ,DateOfBirth
  ,Gender
  ,PermanentAddressLine1
  ,PhoneNumber
  ,PlanMemberID
  ,BeneficiaryID
  ,UniquePersonKey
  ,sha2(concat_ws('|',IfNull(BISInternalPersonID,''),IfNull(IsCurrent,''),IfNull(UniqueRecord,''),IfNull(CAST(FileLayoutID AS STRING),''),IfNull(CAST(FileId AS STRING),''),IfNull(LastName,''),IfNull(FirstName,''),IfNull(DateOfBirth,''),IfNull(Gender,''),IfNull(PermanentAddressLine1,''),IfNull(PhoneNumber,''),IfNull(PlanMemberID,''),IfNull(BeneficiaryID,''),IfNull(UniquePersonKey,''),IfNull(CAST(IsCurrentPlanMemberID AS STRING),''),IfNull(CAST(IsCurrentUniquePersonKey AS STRING),''),IfNull(CAST(IsOriginalMemberID AS STRING),''),IfNull(PMUP,''),IfNull(TRY_CAST(IsCurrentPMUP AS STRING),'')), 256) AS hashKey
  ,IsCurrentPlanMemberID
  ,IsCurrentUniquePersonKey
  ,IsOriginalMemberID
  ,PMUP
  ,TRY_CAST(IsCurrentPMUP AS INT) AS IsCurrentPMUP
FROM Final
"""

In [0]:
# Databricks notebook source
# DBTITLE 1,Main execution - Process and write to Gold
logger.info("=" * 60)
logger.info("MAIN EXECUTION STARTED")
logger.info("=" * 60)
print(f"\nProcessing data from: {sourcePath}")
logger.info(f"Processing data from: {sourcePath}")
sparkMem_df = LoadSource(sourcePath)
numRows = sparkMem_df.count()
print(f"Found {numRows} records")
logger.info(f"Total records to process: {numRows}")

if numRows == 0:
    print("No records to process")
    logger.info("No records to process - exiting")
elif numRows == 1:
    print("Single record - skipping linkage, processing directly")
    logger.info("Single record detected - skipping linkage")
    convertedSpark_df = sparkMem_df.withColumn("MatchID", col("UniqueRecord"))
    # Cast numeric columns to proper types (same as multi-record path)
    convertedSpark_df = convertedSpark_df.withColumn("FileID", col("FileID").cast(LongType())).withColumn("RowNumber", col("RowNumber").cast(LongType()))
    
    convertedSpark_df.createOrReplaceTempView("BISCompletePersonTable")
    logger.info("Created temp view: BISCompletePersonTable")
    print("Applying business logic transformations...")
    logger.info("Applying SQL transformations")
    temp_df = spark.sql(finalSQL)
    logger.info(f"SQL transformations completed. Result: {temp_df.count()} records")
    
    if temp_df.filter("IsCurrentPMUP IS NULL").count() > 0:
        logger.error("Validation failed: records contain both PlanMemberID and UniquePersonKey")
        raise Exception("Failed as records contain both a PlanMemberID and UniquePersonKey")
    else:
        print(f"\nWriting to Gold table: {goldTable}")
        logger.info(f"Writing to Gold table: {goldTable}")
        spark.sql(f"DROP TABLE IF EXISTS {goldTable}")
        temp_df.write.format("delta").mode("overwrite").saveAsTable(goldTable)
        print("\n Gold layer processing completed successfully!")
        logger.info("Gold layer processing completed successfully (single record path)")
else:
    print("Running record linkage...")
    logger.info("Multiple records detected - running record linkage")
    try:
        convertedSpark_df = RunLinking(sparkMem_df)
    except Exception as e:
        logger.error(f"RunLinking failed: {str(e)}")
        raise
    convertedSpark_df = convertedSpark_df.withColumn("FileID", col("FileID").cast(LongType())).withColumn("RowNumber", col("RowNumber").cast(LongType()))
    logger.info("Cast FileID and RowNumber to LongType")
    
    convertedSpark_df.createOrReplaceTempView("BISCompletePersonTable")
    logger.info("Created temp view: BISCompletePersonTable")
    print("Applying business logic transformations...")
    logger.info("Applying SQL transformations")
    temp_df = spark.sql(finalSQL)
    logger.info(f"SQL transformations completed. Result: {temp_df.count()} records")
    
    if temp_df.filter("IsCurrentPMUP IS NULL").count() > 0:
        logger.error("Validation failed: records contain both PlanMemberID and UniquePersonKey")
        raise Exception("Failed as records contain both a PlanMemberID and UniquePersonKey")
    else:
        print(f"\nWriting to Gold table: {goldTable}")
        logger.info(f"Writing to Gold table: {goldTable}")
        spark.sql(f"DROP TABLE IF EXISTS {goldTable}")
        temp_df.write.format("delta").mode("overwrite").saveAsTable(goldTable)
        print("\n Gold layer processing completed successfully!")
        logger.info("Gold layer processing completed successfully (multi-record path)")
        logger.info("=" * 60)